# N084 · Dijkstra与0-1 BFS

**目标：** 根据权值范围选择正确的最短路调度规则。

**先修：** N057, N081, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 非负边；松弛；过期堆项；0/1边；deque前后插入；最短路树。

## 从问题到算法

边权不再统一时，FIFO队列不能保证最早出队者距离最短。Dijkstra取当前最小暂定距离，并依赖非负边保证这个距离不会再被绕远路改善。0/1边权可用双端队列把0代价松弛放前端、1代价松弛放后端。负边应明确拒绝，而非照跑后侥幸得到答案。

## 最小实现

**本章接口：** `dijkstra(n, adj, source)`、`zero_one_bfs(n, adj, source)`、`network_delay_time(times, n, k)`

In [1]:
import heapq
from collections import deque

def dijkstra(n,adj,source):
    if any(w<0 for row in adj for _,w in row): raise ValueError('Dijkstra requires nonnegative weights')
    distance=[float('inf')]*n; distance[source]=0; heap=[(0,source)]
    while heap:
        dist,u=heapq.heappop(heap)
        if dist!=distance[u]: continue
        for v,w in adj[u]:
            if dist+w<distance[v]:
                distance[v]=dist+w; heapq.heappush(heap,(distance[v],v))
    return distance

def zero_one_bfs(n,adj,source):
    if any(w not in (0,1) for row in adj for _,w in row): raise ValueError('only 0/1 weights supported')
    distance=[float('inf')]*n; distance[source]=0; queue=deque([(0,source)])
    while queue:
        dist,u=queue.popleft()
        if dist!=distance[u]: continue
        for v,w in adj[u]:
            if dist+w<distance[v]:
                distance[v]=dist+w
                if w==0: queue.appendleft((distance[v],v))
                else: queue.append((distance[v],v))
    return distance

def network_delay_time(times,n,k):
    adj=[[] for _ in range(n)]
    for u,v,w in times: adj[u-1].append((v-1,w))
    maximum=max(dijkstra(n,adj,k-1))
    return -1 if maximum==float('inf') else maximum

## 正确性、前提与复杂度

假设取出的最小暂定距离仍非最优，则更短路径上第一个未确定顶点的前驱已经确定，应已给它不更大的候选，矛盾。非负性保证后缀不能把路径变短。

**代价：** 二叉堆版O((V+E)log(V+E))时间/O(V+E)空间；0-1 BFS为O(V+E)时间。堆中旧条目用距离检查丢弃，不视为新的有效扩展。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

adj=[[(1,1),(2,4)],[(2,1),(3,5)],[(3,1)],[]]
show_table(['顶点','从0的最短距离'],list(enumerate(dijkstra(4,adj,0))))

顶点,从0的最短距离
0,0
1,1
2,2
3,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert network_delay_time([[2,1,1],[2,3,1],[3,4,1]],4,2)==2
assert network_delay_time([[1,2,1]],2,2)==-1
from random import Random
rng=Random(84)
for _ in range(150):
    n=6; edges=[(u,v,rng.randrange(2)) for u in range(n) for v in range(n) if rng.randrange(5)==0]
    adj=[[] for _ in range(n)]
    for u,v,w in edges: adj[u].append((v,w))
    ref=[float('inf')]*n; ref[0]=0
    for _ in range(n-1):
        old=ref[:]
        for u,v,w in edges: ref[v]=min(ref[v],old[u]+w)
    assert dijkstra(n,adj,0)==zero_one_bfs(n,adj,0)==ref
try: dijkstra(2,[[(1,-1)],[]],0)
except ValueError: pass
else: raise AssertionError('negative edge must be rejected')
print('N084: 所有本章断言通过')

N084: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释负边破坏Dijkstra论证。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用0-1 BFS替换堆并比较操作数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 743. Network Delay Time（canonical）
- 1368. Minimum Cost to Make at Least One Valid Path in a Grid（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。